# 01 — Explore the Q12 weights

All 4,192 parameters of microgpt are baked into PL fabric at
synthesis time. They live as 16-bit fixed-point (Q12) values in
`hw/ip/*.hex` and are pulled into BRAM/LUTRAM via `$readmemh` in
`microgpt_exact_core_rom_init.svh`.

This notebook loads the hex files, decodes them as Q12 fixed-point,
and renders each weight tensor so you can see the structure of the
**actual** model that ends up in the gates.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

IP_DIR = Path('../hw/ip').resolve()

FRAC_BITS = 12   # Q12: 1 sign + 3 integer + 12 fractional bits → range [-8, 8)

WEIGHTS = [
    ('wte_q12.hex',            'WTE — token embedding',        (27, 16)),
    ('wpe_q12.hex',            'WPE — positional embedding',   (16, 16)),
    ('layer0_attn_wq_q12.hex', 'W_Q — attention query',        (16, 16)),
    ('layer0_attn_wk_q12.hex', 'W_K — attention key',          (16, 16)),
    ('layer0_attn_wv_q12.hex', 'W_V — attention value',        (16, 16)),
    ('layer0_attn_wo_q12.hex', 'W_O — attention output',       (16, 16)),
    ('layer0_mlp_fc1_q12.hex', 'FC1 — MLP up-projection',      (16, 64)),
    ('layer0_mlp_fc2_q12.hex', 'FC2 — MLP down-projection',    (64, 16)),
    ('lm_head_q12.hex',        'LM head — logits projection',  (16, 27)),
]


## Decoder

Each `.hex` line is a 16-bit two's-complement word. We convert to a
Python `int`, take the signed value, and divide by `2**FRAC_BITS` to
get the real-valued weight.


In [ ]:
def load_q12_hex(path, shape):
    raw = np.array([int(l.strip(), 16) for l in path.read_text().splitlines() if l.strip()], dtype=np.uint16)
    signed = raw.astype(np.int32)
    signed[signed >= 0x8000] -= 0x10000   # two's-complement → signed
    fp = signed.astype(np.float64) / (1 << FRAC_BITS)
    assert fp.size == shape[0] * shape[1], f'{path.name}: expected {shape}, got {fp.size}'
    return fp.reshape(shape)

tensors = {label: load_q12_hex(IP_DIR / fname, shape) for fname, label, shape in WEIGHTS}
total_params = sum(t.size for t in tensors.values())
print(f'Loaded {len(tensors)} tensors · {total_params} parameters total')


## Visualise

Heatmap each tensor with a diverging colormap centred at zero.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 11), constrained_layout=True)
for ax, (label, w) in zip(axes.flat, tensors.items()):
    vmax = float(np.abs(w).max()) or 1e-9
    ax.imshow(w, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto', interpolation='nearest')
    ax.set_title(f'{label}\n{w.shape} · |w|≤{vmax:.2f} · σ={w.std():.3f}', fontsize=9)
    ax.set_axis_off()
plt.show()


## What you should see

- **WTE / WPE** have visible per-token / per-position structure.
- **W_Q, W_K, W_V** are 16×16 matrices that the systolic matvec
  tile multiplies the embedded token by, every step.
- **FC1 (16→64)** and **FC2 (64→16)** are the MLP block.
- **LM head (16→27)** projects to vocabulary logits.

Every one of these values is baked into LUTRAM / BRAM at synth time —
there is no DRAM behind any of it. If you change a weight, you have
to rebuild the bitstream.

## Sanity check vs the build artefacts


In [ ]:
# Histogram of the LM-head weights — should be roughly centred
import numpy as np
lm = tensors['LM head — logits projection']
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(lm.flatten(), bins=40, color='steelblue', edgecolor='black')
ax.set_xlabel('Q12 weight value'); ax.set_ylabel('count')
ax.set_title(f'lm_head weights · n={lm.size} · σ={lm.std():.3f}')
plt.show()
